# Preprocessing
Here, we are pulling the data to use to create the NLP text corpus. 
To create the text corpus, I pulled text from 214 bills passed to 
law between 1973 and 2025. These were found using the following filters:
- type: Legislation
- passage: Became Law
- policy area: Environmental Policy
- years: 1973-2025

balance and lack of storage.

First, we import the needed packages.

In [1]:
import re
import requests
from io import BytesIO
import pandas as pd
import pypdf
from bs4 import BeautifulSoup
# word processing !
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
import string

# coutner
from collections import Counter # HINT

# topic modelling
import math
import numpy as np


To use the congress.gov site, we need to create an API key. This lets us access the
data on their web server!

In [2]:
API_KEY = "0hq2u14L9UXrF447WVgvPVPJAtgZogwxabCzJaWW"

This enables us to start parsing bills. First, we write parse_bill_url. 
This takes in the url of a bill and returns all of the data associated
with the bill in question from the url.

The format for getting our summary is the following: 
bill/{congress number}/{bill type}/{bill number}/summaries

bill_type is either "hr" or "s", depending on if it is from the house or the 
senate, respectively.

In [3]:
# translate between url form of bill type + shortened bill type
bill_type_map = {
    "house-bill": "hr",
    "senate-bill": "s",
    "house-resolution": "hres",
    "senate-resolution": "sres",
    "house-joint-resolution": "hjres",
    "senate-joint-resolution": "sjres",
    "house-concurrent-resolution": "hconres",
    "senate-concurrent-resolution": "sconres",
}


Create a parsing function to get the relevant information from the url.

In [4]:
def parse_url(url):
    split = url.split('/')
    # get the congress type. 
    congress = int(split[4].split('-')[0][:-2])
    # get bill type and number
    bill_type = bill_type_map[split[5]]
    bill_num = int(split[6])
    
    
    return congress, bill_type, bill_num

In [5]:
#test it! 
# expected output: 100, s, 2030
parse_url('https://www.congress.gov/bill/100th-congress/senate-bill/2030')

(100, 's', 2030)

Now, we can get the bill summary using these isolated elements. 

In [5]:
def get_summary(url):
    congress, bill_type, bill_num = parse_url(url)
    url = f"https://api.congress.gov/v3/bill/{congress}/{bill_type}/{bill_num}/text"
    params = {"api_key": API_KEY, "format": "json"}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()['textVersions'][0]['formats'][0]['url']

In [7]:
# test it!
get_summary('https://www.congress.gov/bill/93rd-congress/senate-bill/433')

'https://www.congress.gov/93/statute/STATUTE-88/STATUTE-88-Pg1660-2.pdf'

Yasss. This gives us all of our urls for text mining. 🕺🕺
Now, we can apply this to our legislation sheet to get the pdfs!

In [6]:
legislation = pd.read_excel('Legislation.xlsx')

In [7]:
legislation['text'] = legislation.apply(lambda row: get_summary(row['URL']), axis=1)

#### Extracting text from pdfs
Now that we have the links to all of the legislative pdfs, we can now extract
the full text ! To do this, we are using the pypdf package. This will parse
the text from each pdf into streams we can use for NLP.

In [8]:
from wordfreq import zipf_frequency

# pdf cleaning
def _looks_like_word(word: str, min_zipf: float = 2.0) -> bool:
    """
    Returns True if a token looks like a real English word.
    If wordfreq is unavailable, fall back to a conservative heuristic.
    """
    word = word.lower()

    if not word.isalpha():
        return False

    if zipf_frequency is not None:
        return zipf_frequency(word, "en") >= min_zipf

    # Fallback heuristic if wordfreq is not installed
    return len(word) >= 3


def _repair_fragmented_words(text: str) -> str:
    """
    Repairs tokens split into adjacent fragments, e.g.
    'guidanc e' -> 'guidance'
    'federa l' -> 'federal'
    'propos ed' -> 'proposed'
    """
    tokens = text.split()
    repaired = []
    i = 0

    while i < len(tokens):
        tok = tokens[i]

        if i + 1 < len(tokens):
            nxt = tokens[i + 1]

            if tok.isalpha() and nxt.isalpha():
                merged = tok + nxt

                # join short trailing fragment if merged form looks valid
                if len(tok) >= 4 and len(nxt) <= 3 and _looks_like_word(merged):
                    repaired.append(merged)
                    i += 2
                    continue

        repaired.append(tok)
        i += 1

    return " ".join(repaired)


def _split_merged_word(token: str) -> str:
    """
    Attempts to split OCR-merged words like:
    'federaldepart' -> 'federal depart'
    'andth' stays unchanged unless a plausible split is found.
    """
    if not token.isalpha() or len(token) < 8:
        return token

    for j in range(3, len(token) - 2):
        left = token[:j]
        right = token[j:]

        if _looks_like_word(left, 3.0) and _looks_like_word(right, 3.0):
            return left + " " + right

    return token


def _repair_merged_words(text: str) -> str:
    return " ".join(_split_merged_word(tok) for tok in text.split())


def _normalize_legal_citations(text: str) -> str:
    """
    Normalizes common legal citation patterns.
    """
    text = re.sub(
        r'\b(\d+)\s+usc\s+([0-9A-Za-z\-]+)\b',
        r'\1 U.S.C. \2',
        text,
        flags=re.I
    )
    text = re.sub(
        r'\b(\d+)\s+[Ss]tat[.,]?\s+(\d+)\b',
        r'\1 Stat. \2',
        text
    )
    text = re.sub(
        r'\b[Pp]ublic\s+[Ll]aw\s+(\d+)\s*-\s*(\d+)\b',
        r'Public Law \1-\2',
        text
    )
    return text


def readPdf(url):
    """
    Reads a PDF from a URL and extracts text page by page.
    Keeps page separators so cleanup does not accidentally fuse pages.
    """
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()

    pdf_file = BytesIO(resp.content)
    reader = pypdf.PdfReader(pdf_file)

    pages = []

    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text(
            extraction_mode="layout",
            layout_mode_space_vertically=True
        ) or ""

        pages.append(f"\n\n--- PAGE {i} ---\n\n{text}")

    return "".join(pages)


def clean_pdf_text(text, remove_numbers=True):
    """
    Cleans extracted legislative PDF text for downstream NLP/topic modeling.

    Main steps:
    1. normalize line endings / soft hyphens
    2. remove page/header/footer noise
    3. repair line-break word fragmentation
    4. join wrapped prose lines carefully
    5. repair split and merged OCR tokens
    6. normalize legal citations
    7. optionally remove numbers
    """
    # ------------------------------------------------------------------
    # 1) Normalize raw text
    # ------------------------------------------------------------------
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\xad", "")   # soft hyphen
    text = text.replace("\u00ad", "") # redundant explicit soft hyphen form

    # ------------------------------------------------------------------
    # 2) Remove page markers and common legislative header/footer noise
    # ------------------------------------------------------------------
    text = re.sub(r"\n?--- PAGE \d+ ---\n?", "\n\n", text)

    text = re.sub(r'(?im)^\s*\d+\s+STAT\..*$', '', text)
    text = re.sub(r'(?im)^\s*PUBLIC LAW\s+\d+\s*-\s*\d+.*$', '', text)
    text = re.sub(r'(?im)^\s*Approved\s+[A-Z][a-z]+\s+\d{1,2},\s+\d{4}\.?\s*$', '', text)
    text = re.sub(r'(?im)^\s*\[\s*[A-Z]\.\s*R\.\s*\d+\s*\]\s*$', '', text)
    text = re.sub(r'(?im)^\s*\d+\s*$', '', text)  # bare page number lines

    # ------------------------------------------------------------------
    # 3) Repair fragmented words created by line wrapping
    # ------------------------------------------------------------------
    # hyphenated line-wraps: "subsectio-\nn" -> "subsection"
    text = re.sub(r'(?<=\w)-\n(?=\w)', '', text)

    # split words without hyphen across newlines: "fisca\nl" -> "fiscal"
    text = re.sub(r'(?<=[a-z])\n(?=[a-z])', '', text)

    # one-letter or short fragment splits with spaces: "fisca l" -> "fiscal"
    # do this conservatively before broader normalization
    text = re.sub(r'\b([A-Za-z]{4,})\s([A-Za-z]{1,2})\b', r'\1\2', text)

    # ------------------------------------------------------------------
    # 4) Join wrapped prose lines, but not everything
    # ------------------------------------------------------------------
    text = re.sub(r'\n{3,}', '\n\n', text)

    # join likely sentence continuations
    text = re.sub(r'(?<=[a-z,;:])\n(?=[a-z("\'])', ' ', text)

    # join lines where the next line clearly continues prose
    text = re.sub(r'(?<!\n)\n(?=[a-z0-9"\')\],;:])', ' ', text)

    # ------------------------------------------------------------------
    # 5) Normalize whitespace before token-level repair
    # ------------------------------------------------------------------
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r' *\n *', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()

    # ------------------------------------------------------------------
    # 6) Token-level OCR repair
    # ------------------------------------------------------------------
    text = _repair_fragmented_words(text)
    text = _repair_merged_words(text)

    # ------------------------------------------------------------------
    # 7) Normalize common legal citations / OCR substitutions
    # ------------------------------------------------------------------
    replacements = {
        " use ": " usc ",
        " USe ": " USC ",
        " stat, ": " Stat. ",
        "''": '"',
        "``": '"',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    text = _normalize_legal_citations(text)

    # ------------------------------------------------------------------
    # 8) Remove numbers if desired (best for topic modeling)
    # ------------------------------------------------------------------
    if remove_numbers:
        # remove money amounts, years, section numbers, dates, numeric tokens
        text = re.sub(r'\$?\b\d[\d,.\-]*\b', ' ', text)

        # remove month names if you do not want date-heavy topics
        text = re.sub(
            r'\b(?:jan|january|feb|february|mar|march|apr|april|may|jun|june|jul|july|aug|august|sep|sept|september|oct|october|nov|november|dec|december)\.?\b',
            ' ',
            text,
            flags=re.I
        )

    # ------------------------------------------------------------------
    # 9) Final cleanup
    # ------------------------------------------------------------------
    text = re.sub(r'[^A-Za-z\s\.\,\;\:\(\)\-"]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [9]:
# htm cleaning
def readHtm(url):
    resp = requests.get(url)
    if resp.status_code == 200:
        return resp.text

Now, we can process the rows. 

In [10]:
def process_pdf(row):
    date = row['date']
    name = row['Title']

    text = readPdf(row['text'])
    clean_text = clean_pdf_text(text)
    return text, clean_text, date, name

After the asbstos act of 1984, they started storing files as htm instead of pdf. okayyy

In [11]:
def process_htm(row):
    date = row['date']
    name = row['Title']

    soup = BeautifulSoup(readHtm(row['text']), "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    
    print(text)
    clean_text = clean_pdf_text(text)
    return text, clean_text, date, name

In [12]:
records = []

### PARSE!!!

In [17]:
for i, row in legislation.iterrows():
    if i < 97:
        text, clean, date, name = process_pdf(row)
        
        records.append({
            "text_raw": text,
            "text_clean": clean,
            "date_raw": date,
            "date_parsed": name
        })
        print(f"{name} parsed.")
    
    # parsing htm....
    else:
        text, clean, date, name = process_htm(row)
        
        records.append({
            "text_raw": text,
            "text_clean": clean,
            "date_raw": date,
            "date_parsed": name
        })
        print(f"{name} parsed.")
    

ConnectionError: HTTPSConnectionPool(host='www.congress.gov', port=443): Max retries exceeded with url: /93/statute/STATUTE-88/STATUTE-88-Pg1660-2.pdf (Caused by NameResolutionError("HTTPSConnection(host='www.congress.gov', port=443): Failed to resolve 'www.congress.gov' ([Errno 8] nodename nor servname provided, or not known)"))

### Topic Modelling

That gives us our full text corpus. Hell yes. HELL YES!!!
Now, we need to process the text to a useable form!

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/skylarwalters/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/skylarwalters/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/skylarwalters/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/skylarwalters/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
def tokenize(text):
    return word_tokenize(text)

def punct(text):
    return [word for word in text if word not in string.punctuation]

def stem_and_stop(text):
    stop_words = set(stopwords.words('english'))
    stopped = [word for word in text if word not in stop_words]

    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(word) for word in stopped]
    return stemmed

def lemmatize(text):
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(word) for word in text]
    return lemmatized

In [97]:
def process_text(text):
    return lemmatize(stem_and_stop(punct(tokenize(text))))

In [ ]:
def create_corpus(records):
    idx_to_year = {}
    idx_to_act = {}
    corpus =  []

    for i, record in enumerate(records):
        idx_to_year[i] = record['date_raw']
        idx_to_act[i] = record['date_parsed']
        
        corpus.append(process_text(record['text_clean']))
        
    return corpus, idx_to_year, idx_to_act
        

In [ ]:
corpus, idx_to_year, idx_to_act = create_corpus(records)

In [ ]:
len(corpus)
len(idx_to_year)

214

### word frequencies

In [ ]:
# make a list of token bt tear

EPICCCC. Now to do the rest of the processing. This code was taken from csci1460.

## Topic Modelling

In [101]:
def create_vocab(corp : list[list[str]], vocab_cutoff : int = 2500) -> list[str]:
    """
    Aggregates and collects the text of the most common tokens in docs, cutoff by the vocab_cutoff.

    Parameters
    ----------
    proc_docs : list[list[str]]
        A list of preprocessed documents using preprocess_doc
    vocab_cutoff : int
        The cutoff of the MOST FREQUENT vocabulary

    Returns
    -------
    vocab : list[str]
        A list of the top {vocab_cutoff} most common tokens in proc_docs
    """
    counts = Counter()
    for doc in corp:
      counts.update(doc)

    return [word for word, _ in counts.most_common(vocab_cutoff)]

In [40]:
def idf_matrix(proc_docs : list[list[str]], word2idx : dict[str, int],  vocab : list[str]) -> np.ndarray[np.float64]:
    """
    Calculates the inverse document frequency (idf) matrix using the equation above for each word
    Equation: idf[w] = ln(|D| / |d in D : w in d,v|)

    Parameters
    ----------
    proc_docs : list[list[str]]
        A list of preprocessed documents using preprocess_doc
    word2idx : dict[str, int]
        A dictionary that matches each word in the vocabulary to its rank
    vocab : list[str]
        Vocab of all the docs (thresholded by vocab_cutoff)

    Returns
    -------
    idf : np.array[float]
        idf array, as defined by the equation in the description
    """
    # TODO

    # HINT: It may be useful to start by counting the number of documents a word shows up in for each word
    # dimension: num of words
    idf = np.zeros(len(vocab))
    num_docs = len(proc_docs)

    # count number of docs each word is in.
    word2numDoc = {}
    for doc in proc_docs:
      docSet = set(doc)

      for item in list(docSet):
        if item not in word2numDoc:
          word2numDoc[item] = 1
        else:
          word2numDoc[item] += 1

    # iterate through the vocab and put the words into the idf matrix
    for word in vocab:
      idf[word2idx[word]] = math.log(num_docs / word2numDoc[word])

    return idf

In [41]:
def tf_matrix(proc_docs : list[list[str]], word2idx : dict[str, int], vocab : list[str]) -> np.ndarray[np.float64]:
    """
    Calculates the term frequency (tf) matrix using the equation above for each word
    Equation: tf[w, d] = 0.5 + 0.5 * (freq_w_in_d / freq_wmax_in_d_and_v)

    Parameters
    ----------
    proc_docs : list[list[str]]
        A list of preprocessed documents using preprocess_doc
    word2idx : dict[str, int]
        A dictionary that matches each word in the vocabulary to its rank
    vocab : list[str]
        Vocab of all the docs (thresholded by vocab_cutoff)

    Returns
    -------
    tf : np.array[Float]
        tf array, as defined by the equation in the description
    """
    # TODO
    # NOTE: For words not in the document, the tf value should be 0, NOT 0.5
    # HINT: Try to also think of edge cases and consider all possible document types in your implementation
    # check if num rows or num cols is the num docs

    tf = np.zeros((len(proc_docs), len(vocab)), dtype=np.float64) # CAST TO FLOAT64!

    for d in range(len(proc_docs)):

      # filter non-vocab words
      doc = [word for word in proc_docs[d] if word in word2idx]
      w_freq = Counter(doc)

      for w in w_freq.keys():
       # if w in word2idx:
        tf[d][word2idx[w]] = 0.5 + 0.5 * (w_freq[w] / w_freq.most_common(1)[0][1])

    return tf

In [42]:
def tfidf_term_doc_matrix(corp, vocab_cutoff : int = 5000) -> tuple[np.ndarray[np.float64], dict[int, str]]:
    """
    Computes a term-document matrix M using tf-idf, as well as a dictionary mapping
    each word in the vocab's index/rank to the word itself

    Parameters
    ----------
    docs : list[spacy_doc]
        A list of spaCy preprocessed documents
    vocab_cutoff : int
        The cutoff of the MOST FREQUENT vocabulary

    Returns
    -------
    M : np.ndarray[float]
        The tf-idf term document matrix
    idx2word : dict[int, str]
        The dictionary that maps each index/rank to each word in the vocabulary
    """
    # TODO: There are multiple steps in this function:
    #       1. Preprocess each document and compute your thresholded vocab
    #       2. Fill in word2idx and idx2word
    #       3. Compute the tf and idf arrays
    #       4. Combine the tf and idf arrays to make the tf-idf array (M)

    # preproc the docs. predoccess. if you will. also create vocabulary
    #proc_docs = [preprocess_doc(doc) for doc in docs] # list list str output
    vocab = create_vocab(corp, vocab_cutoff)

    # populate dicts. lowkey we do not need to rank these...
    word2idx = {word: ind for ind, word in enumerate(vocab)}
    idx2word = {ind: word for ind, word in enumerate(vocab)}

    tf = tf_matrix(corp, word2idx, vocab)
    idf = idf_matrix(corp, word2idx, vocab)

    M = np.zeros((len(corp), len(vocab)))
    M = tf * idf.reshape(1, len(vocab))

    return M, idx2word

In [29]:
import sklearn
from sklearn.decomposition import LatentDirichletAllocation

def train_topic_model(term_doc_mat : np.ndarray[np.float64], n_topics : int = 5, random_state = 42) -> LatentDirichletAllocation:
    """
    Train a n_topics topic model on M using latent Dirichlet allocation

    Parameters
    ----------
    term_doc_mat : np.ndarray[float]
        A term-document matrix to train the LDA model on
    n_topics : int
        Number of topics in the topic model (default: 10)
    random_state : int
        A random state of the LDA nodel (default: 42)

    Returns
    -------
    lda : LatentDirichletAllocation
        A trained LDA model
    """

    # TODO: Train an LDA model with n_topics on the given term-document matrix, then return the model.
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=random_state)

    return lda.fit(term_doc_mat)
    # NOTE: You MUST use the given random state when intialzing your LDA model.

In [30]:
def preview_topics(topic_model: LatentDirichletAllocation, idx2word: dict[int, str]) -> list[list[str]]:
    """
    Returns a list of the 10 words most associated with each topic from the trained topic model

    Parameters
    ----------
    topic_model : LatentDirichletAllocation
        A trained LDA topic model
    idx2word : dict[int, str]
        A dictionary that maps each index/rank to each word in the vocabulary

    Returns
    -------
    topic_words : list[list[str]]
        A list of the 10 words associated with each topic
    """
    # TODO: Return the top 10 words associated with each topic
    # HINT: You will need to use idx2word and will likely find numpy's argsort to be helpful here.
    #       Make sure you check the sklearn documentation on how to get each topic from the model.
    topics = topic_model.components_
    outs = []

    for entry in topics:
      # we now have a list of many parameters.
      # we need to find the top 10 highest numbers from this list
      # map the indices of these numbers to the word! using idx2word.
      most = np.argsort(entry)[-10:]

      outs.append([idx2word[index] for index in most])

    return outs

In [102]:
M, idx2word = tfidf_term_doc_matrix(corpus)
topic_model = train_topic_model(M, n_topics=5)
top_topics = preview_topics(topic_model, idx2word)
for i in range(5):
    print(f"Topic {i+1}: {top_topics[i]}")

Topic 1: ['cityof', 'heldat', 'washingtonon', 'u.', 'bill', 'enrol', 'america', 'enr', 'general.', '--']
Topic 2: ['issuean', 'judici', 'pursuant', 'crimin', 'fine', 'deni', 'penal', 'commenc', 'penalti', 'evid']
Topic 3: ['chant', 'mer', 'comm', 'pas', 'hous', 'insertingin', 'histori', 'lieu', 'vol', 'reportno']
Topic 4: ['resolutionto', 'peopleof', 'resolvedbi', 'therefor', 'proclam', 'requestedto', 'call', 'ceremoni', 'now', 'wherea']
Topic 5: ['joint', 'friday', 'fed', 'haveno', 'reg', 'resolut', 'forceor', 're', 'resolvedbi', 'h.j']


Now, to topic model by decade.

In [32]:
records[0]['date_raw'].year

1973

In [33]:
seventy = []
e_80 = []
l_80 = []
e_90 = []
l_90 = []
e_00 = []
l_00 = []
e_10 = []
l_10 = []
e_20 = []

In [34]:
for ind in idx_to_year:
    yr = idx_to_year[ind]
    if yr.year < 1980:
        seventy += [ind]
    elif yr.year <1985:
        e_80 += [ind]
    elif yr.year <1990:
        l_80 += [ind]
    elif yr.year <1995:
        e_90 += [ind]
    elif yr.year <2000:
        l_90 += [ind]
    elif yr.year <2005:
        e_00 += [ind]
    elif yr.year <2010:
        l_00 += [ind]
    elif yr.year <2015:
        e_10 += [ind]
    elif yr.year <2020:
        l_10 += [ind]
    else:
        e_20 += [ind]

In [140]:
e_20

[197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 207,
 208,
 209,
 210,
 211,
 212,
 213]

In [60]:
M, idx2word = tfidf_term_doc_matrix(corpus[108:137])
topic_model = train_topic_model(M, n_topics=5)
top_topics = preview_topics(topic_model, idx2word)
for i in range(5):
    print(f"Topic {i+1}: {top_topics[i]}")

Topic 1: ['treat', 'fill', 'solid', 'wast', 'perform', 'federalor', 'inciner', 'tract', 'separ', 'recycl']
Topic 2: ['brown', 'read', 'whether', 'least', 'under', 'district', 'cd', 'semicolon', 'citizen', 'lake']
Topic 3: ['pacif', 'boundariesof', 'boundaryof', 'basin', 'west', 'areasin', 'sanctuari', 'sea', 'boundari', 'fisheri']
Topic 4: ['foundationto', 'select', 'expens', 'truste', 'award', 'bequest', 'bear', 'gift', 'student', 'foundat']
Topic 5: ['fl-', 'reclam', 'suspend', 'diego', 'stamp', '``', 'general.', 'duck', 'correctionin', 'p']


### Embed shifts

In [139]:
from gensim.models import Word2Vec

# Shared hyperparameters for ALL slices
w2v_params = dict(
    vector_size=200,   # embedding dimension
    window=5,          # context window
    min_count=5,       # min frequency threshold
    workers=4,         # adjust for your machine
    sg=1,              # 1 = skip-gram, 0 = CBOW
    negative=10,       # negative samples
    epochs=10,         # training epochs
    seed=42
)

In [141]:
models_by_slice = {}
incr =[0,44,72,108,137,152,161,172,182,197,214]

for i in range(0,len(incr) - 1):    
    model = Word2Vec(**w2v_params)
    # Build vocabulary from this slice
    model.build_vocab(corpus[incr[i]:incr[i + 1]])
    model.train(
        corpus[incr[i]:incr[i + 1]],
        total_examples=model.corpus_count,
        epochs=model.epochs
    )
    models_by_slice[i] = model

In [230]:
v1 = set(models_by_slice[0].wv.key_to_index.keys())
v2 = set(models_by_slice[1].wv.key_to_index.keys())
v3 = set(models_by_slice[2].wv.key_to_index.keys())
v4 = set(models_by_slice[3].wv.key_to_index.keys())
v5 = set(models_by_slice[4].wv.key_to_index.keys())
v6 = set(models_by_slice[5].wv.key_to_index.keys())
v7 = set(models_by_slice[6].wv.key_to_index.keys())
v8 = set(models_by_slice[7].wv.key_to_index.keys())
v9 = set(models_by_slice[8].wv.key_to_index.keys())
v10= set(models_by_slice[9].wv.key_to_index.keys())
sets = [v2, v3, v4, v5, v6, v7, v8]
shared = v1.intersection(*sets)
shared
v9

{'final',
 'regul',
 'differ',
 'con',
 'establishment.',
 'applicant-submit',
 'expirationof',
 'scientifically-sound',
 'complex',
 'prescribedbi',
 'epa',
 'institut',
 'individu',
 'withan',
 'expertis',
 'suffici',
 'exposur',
 'find',
 'modifi',
 'dome',
 'post-consum',
 'sourcesof',
 'bug',
 'servic',
 'control',
 'estuari',
 'lie',
 'if',
 'agenc',
 'th',
 'from',
 'continu',
 'duct',
 'non-food',
 'pack',
 'expir',
 'codeof',
 'defici',
 'listof',
 'applicationto',
 'determinedbi',
 'least',
 'invas',
 'mitig',
 'qualiti',
 'amendmentsto',
 'exclus',
 'semicolon',
 'dateon',
 'epa-initi',
 'identifi',
 'unless',
 'changeto',
 'everi',
 'agre',
 'economi',
 'chemic',
 'stateor',
 'dead',
 'excepta',
 'intoa',
 'committeeon',
 'termof',
 'lossof',
 'data',
 'bullet',
 'expo',
 'availabilityof',
 'graphic',
 'specif',
 'futur',
 'conclud',
 'gui',
 'assert',
 'usc',
 'residu',
 'transport',
 'sub',
 'drink',
 'sa',
 'product',
 'certifi',
 'carri',
 'identifiedbi',
 'subsequ',
 '

In [163]:
def print_similar(doc):
    for pair in doc:
        print(pair[0])
    

In [232]:
for i in range(0, 10):
    print(i)
    try:
        print_similar(models_by_slice[i].wv.most_similar("pipelin", topn=10))
    except:
        continue


0
spill
subsoil
incid
or
discoveryof
gear
onshor
convert
craft
offshor
1
2
3
hazardto
operatorsof
termi
operatorof
facilitiesin
excav
maximum
inspectionof
facil
valv
4
5
6
7
8
9


In [2]:
models_by_slice[1].wv.most_similar("pipelin", topn=10)

NameError: name 'models_by_slice' is not defined